# Handling Imbalanced Dataset

---

# 1. What problem does it solve?

An **imbalanced dataset** is one where one class has significantly more samples than another.

Example:

| Class    | Count |
| -------- | ----: |
| No Fraud | 9,900 |
| Fraud    |   100 |

A model trained on this data may predict **"No Fraud"** for every transaction and still achieve **99% accuracy**, while completely failing to detect fraud.

### Problems caused by imbalance

* Misleadingly high accuracy
* Poor detection of the minority class
* Low Recall and F1-score
* Biased predictions toward the majority class
* Important events (fraud, disease, defects) are missed

---

# 2. Why do imbalanced datasets occur?

Imbalance is common in real-world problems because rare events naturally occur less frequently.

Examples:

* Fraud transactions are rare.
* Cancer cases are fewer than healthy cases.
* Machine failures happen infrequently.
* Manufacturing defects are uncommon.
* Spam emails are fewer than legitimate emails (depending on the dataset).

---

# 3. When should I use imbalance handling?

Use it when:

* One class is much smaller than another (e.g., 95:5, 99:1).
* You care about correctly identifying the minority class.
* Accuracy alone is misleading.

Typical applications:

* Fraud detection
* Medical diagnosis
* Intrusion detection
* Credit default prediction
* Defect detection

---

# 4. When should I NOT use it?

* When classes are already reasonably balanced.
* For regression problems (imbalance handling is mainly for classification).
* If imbalance reflects business priorities and resampling would distort the data unnecessarily.
* Don't apply resampling before splitting into train and test sets (causes data leakage).

---

# 5. Types of Imbalance

## A. Mild Imbalance

Example:

| Class | Count |
| ----- | ----: |
| A     |   600 |
| B     |   400 |

Usually manageable without special techniques.

---

## B. Moderate Imbalance

Example:

| Class | Count |
| ----- | ----: |
| A     |   900 |
| B     |   100 |

May require class weights or resampling.

---

## C. Severe Imbalance

Example:

| Class | Count |
| ----- | ----: |
| A     | 9,900 |
| B     |   100 |

Often requires specialized techniques like SMOTE or cost-sensitive learning.

---

# 6. Methods

## Method 1: Random Under-Sampling

Remove samples from the majority class.

### Example

Before:

| Majority | Minority |
| -------: | -------: |
|      900 |      100 |

After:

| Majority | Minority |
| -------: | -------: |
|      100 |      100 |

### Pros

* Fast.
* Reduces training time.

### Cons

* Loses potentially useful information.
* Can reduce model performance.

### Code

```python
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)
```

---

## Method 2: Random Over-Sampling

Duplicate minority-class samples.

### Example

Before:

| Majority | Minority |
| -------: | -------: |
|      900 |      100 |

After:

| Majority | Minority |
| -------: | -------: |
|      900 |      900 |

### Pros

* No information loss.
* Easy to implement.

### Cons

* Can cause overfitting because duplicate samples are repeated.

### Code

```python
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
```

---

## Method 3: Class Weights

Instead of changing the data, assign a higher penalty to mistakes on the minority class.

Example:

* Majority class weight = 1
* Minority class weight = 10

### Best for

* Logistic Regression
* SVM
* Decision Trees
* Random Forest

### Code

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced")
```

### Pros

* No duplication.
* No data loss.
* Simple to use.

### Cons

* Doesn't create new information.

---

## Method 4: Threshold Tuning

By default, many classifiers predict the positive class if its probability is ≥ 0.5.

Example:

Instead of:

```
Probability > 0.50
```

Use:

```
Probability > 0.30
```

This often increases Recall for the minority class, at the cost of more false positives.

---

## Method 5: Ensemble Methods

Algorithms designed to improve performance on imbalanced datasets.

Examples:

* Balanced Random Forest
* EasyEnsemble
* RUSBoost

These combine sampling with ensemble learning.

---

## Method 6: SMOTE (Synthetic Minority Over-sampling Technique)

Instead of duplicating minority samples, SMOTE generates **new synthetic samples** by interpolating between existing minority-class points.

This is one of the most widely used techniques and is usually covered as a separate topic.

---

# 7. Decision Flow

```text
Classification Problem?

│

└── Yes

      │

      ▼

Class Distribution Balanced?

│

├── Yes → Train Normally

│

└── No

      │

      ▼

Minority Class Important?

│

├── No → Accuracy may be sufficient

│

└── Yes

      │

      ▼

Small Dataset?

├── Yes → Over-Sampling / SMOTE

└── No

      │

      ▼

Large Majority Class?

├── Yes → Under-Sampling

└── No

      │

      ▼

Algorithm Supports Class Weights?

├── Yes → Use Class Weights

└── Otherwise → Resampling + Model Evaluation
```

---

# 8. Code (Scikit-learn + imbalanced-learn)

```python
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
```

> **Important:** Always split the data first, then apply any resampling only to the training set.

---

# 9. Pros & Cons

| Method           | Pros                      | Cons                                                 |
| ---------------- | ------------------------- | ---------------------------------------------------- |
| Under-Sampling   | Fast, smaller dataset     | Loses information                                    |
| Over-Sampling    | Keeps all original data   | Can overfit                                          |
| Class Weights    | No data modification      | May not be enough for severe imbalance               |
| Threshold Tuning | Improves Recall           | More false positives                                 |
| SMOTE            | Creates synthetic samples | Can create unrealistic samples near class boundaries |
| Ensemble Methods | Strong performance        | More computationally expensive                       |

---

# 10. Algorithm Compatibility

| Algorithm           | Can Use Class Weights?                     |
| ------------------- | ------------------------------------------ |
| Logistic Regression | ✅                                          |
| SVM                 | ✅                                          |
| Decision Tree       | ✅                                          |
| Random Forest       | ✅                                          |
| XGBoost             | ✅ (`scale_pos_weight`)                     |
| LightGBM            | ✅ (`is_unbalance` / `scale_pos_weight`)    |
| CatBoost            | ✅                                          |
| KNN                 | ❌ (typically relies on resampling instead) |
| Naive Bayes         | ❌ (generally use resampling)               |

---

# 11. Evaluation Metrics for Imbalanced Data

**Avoid relying only on accuracy.**

Instead, focus on:

* Precision
* Recall
* F1-Score
* ROC-AUC
* PR-AUC (Precision-Recall AUC), especially for highly imbalanced datasets
* Confusion Matrix

Example:

| Metric   | Value |
| -------- | ----: |
| Accuracy |   99% |
| Recall   |   12% |

Even with 99% accuracy, the model performs poorly because it misses most positive cases.

---

# 12. Interview Questions

### Basic

* What is an imbalanced dataset?
* Why is accuracy misleading on imbalanced data?
* What is Recall?

### Intermediate

* Difference between over-sampling and under-sampling?
* Why use class weights?
* When should you prefer Precision over Recall?

### Advanced

* How does SMOTE work?
* What are the limitations of SMOTE?
* Why should resampling only be done on the training data?
* How would you evaluate a fraud detection model?

---

# 13. Common Mistakes

❌ Judging performance only by accuracy.

❌ Applying over-sampling or under-sampling before train-test splitting (data leakage).

❌ Using SMOTE on the test set.

❌ Ignoring Precision and Recall.

❌ Assuming every imbalanced dataset requires SMOTE.

❌ Forgetting to use stratified train-test splitting for classification tasks.

---

# 14. Real-World Examples

### Banking

Credit card fraud detection:

* 99.8% genuine transactions
* 0.2% fraudulent transactions

Goal: Maximize Recall while keeping false alarms manageable.

---

### Healthcare

Cancer diagnosis:

* Healthy patients greatly outnumber positive cases.

Missing a cancer case (false negative) is usually much more serious than a false alarm.

---

### Manufacturing

Defect detection:

* Thousands of products are defect-free.
* Only a small percentage are defective.

The model should identify rare defects without flagging too many good products.

---

### Cybersecurity

Intrusion detection:

* Most network traffic is legitimate.
* Attacks are relatively rare.

The minority class (attacks) is the one of greatest interest.

---

### Customer Churn

Most customers stay with the company, while relatively few leave.

The business wants to identify customers likely to churn so retention efforts can be targeted.

---

# 15. Revision Box

```text
Handling Imbalanced Dataset

✔ Used for classification problems
✔ Accuracy alone can be misleading

Common Methods:
• Under-Sampling
• Over-Sampling
• Class Weights
• SMOTE
• Threshold Tuning
• Ensemble Methods

Evaluation Metrics:
✔ Precision
✔ Recall
✔ F1-Score
✔ ROC-AUC
✔ PR-AUC
✔ Confusion Matrix

Always:
• Split data first
• Resample only the training set
• Use stratified train-test split
• Compare multiple metrics, not just accuracy

Interview Tip:
For fraud detection or disease diagnosis, high Recall is often more important than high Accuracy.
```


In [13]:
# Import the NumPy library for numerical operations and random number generation
import numpy as np

# Import the Pandas library for creating and manipulating dataframes
import pandas as pd


# Set a fixed random seed so that the random numbers generated are the same
# every time the code is run. This makes the results reproducible.
np.random.seed(123)

# Define the total number of samples (rows) to be created in the dataset.
n_samples = 1000

# Specify the proportion of samples that should belong to Class 0.
# Here, 90% of the dataset will belong to Class 0, creating an imbalanced dataset.
class_0_ratio = 0.9

# Calculate the number of samples for Class 0.
# int() converts the result to an integer since the number of samples
# must be a whole number.
n_class_0 = int(n_samples * class_0_ratio)

# Calculate the number of samples for Class 1.
# This is the remaining 10% of the dataset after allocating samples to Class 0.
n_class_1 = n_samples - n_class_0

In [14]:
# Number of samples belonging to Class 0 and Class 1, respectively
n_class_0, n_class_1

(900, 100)

In [15]:
# Create a DataFrame containing samples for Class 0 (majority class)
class_0 = pd.DataFrame({

    # Generate random values for Feature 1 from a normal distribution
    # with mean (loc) = 0 and standard deviation (scale) = 1.
    # Generate one value for each Class 0 sample.
    'feature_1': np.random.normal(loc=0, scale=1, size=n_class_0),

    # Generate random values for Feature 2 using the same distribution
    # (mean = 0, standard deviation = 1).
    'feature_2': np.random.normal(loc=0, scale=1, size=n_class_0),

    # Assign the target label 0 to all samples in Class 0.
    # [0] * n_class_0 creates a list of zeros of length n_class_0.
    'target': [0] * n_class_0

})

# Create a DataFrame containing samples for Class 1 (minority class)
class_1 = pd.DataFrame({

    # Generate random values for Feature 1 from a normal distribution
    # with mean (loc) = 2 and standard deviation (scale) = 1.
    # The higher mean separates this class from Class 0.
    'feature_1': np.random.normal(loc=2, scale=1, size=n_class_1),

    # Generate random values for Feature 2 using the same distribution
    # (mean = 2, standard deviation = 1).
    'feature_2': np.random.normal(loc=2, scale=1, size=n_class_1),

    # Assign the target label 1 to all samples in Class 1.
    # [1] * n_class_1 creates a list of ones of length n_class_1.
    'target': [1] * n_class_1

})

In [16]:
# Combine the Class 0 and Class 1 DataFrames into a single DataFrame.
# pd.concat() stacks the rows of both DataFrames vertically.
# reset_index(drop=True) resets the row indices to a continuous sequence
# (0, 1, 2, ...) and discards the old indices.
df = pd.concat([class_0, class_1]).reset_index(drop=True)

In [17]:
# see the head
df.head()

,feature_1,feature_2,target
0,-1.085631,0.551302,0
1,0.997345,0.419589,0
2,0.282978,1.815652,0
3,-1.506295,-0.252750,0
4,-0.578600,-0.292004,0


In [18]:
# see the tail
df.tail()

,feature_1,feature_2,target
995,1.376371,2.845701,1
996,2.239810,0.880077,1
997,1.131760,1.640703,1
998,2.902006,0.390305,1
999,2.697490,2.013570,1


In [19]:
# Count the number of occurrences of each class label in the target column.
# This helps verify the class distribution and confirm whether the dataset is imbalanced.
df['target'].value_counts()

,count
target,
0,900
1,100


**Upsampling**

In [20]:
# Create a new DataFrame containing only the minority class samples
# (rows where the target value is 1).
df_minority = df[df['target'] == 1]

# Create a new DataFrame containing only the majority class samples
# (rows where the target value is 0).
df_majority = df[df['target'] == 0]

In [23]:
# Import the resample function from scikit-learn.
# It is used to perform random sampling with or without replacement.
from sklearn.utils import resample

# Upsample the minority class by randomly duplicating its samples.
# replace=True allows the same sample to be selected multiple times.
# n_samples=len(df_majority) increases the minority class size
# to match the number of samples in the majority class.
# random_state=42 ensures reproducible results.
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

In [24]:
df_minority_upsampled

,feature_1,feature_2,target
951,1.125854,1.843917,1
992,2.196570,1.397425,1
914,1.932170,2.998053,1
971,2.272825,3.034197,1
960,2.870056,1.550485,1
...,...,...,...
952,1.188902,2.189189,1
965,3.919526,1.980541,1
976,2.810326,3.604614,1
942,3.621531,2.168229,1


In [27]:
# Combine the original majority class with the upsampled minority class
# to create a balanced dataset containing an equal number of samples
# from both classes.
df_upsampled= pd.concat([df_majority, df_minority_upsampled])

In [31]:
# Count the number of samples in each target class after upsampling.
# This verifies that the dataset is now balanced.
df_upsampled['target'].value_counts()

,count
target,
0,900
1,900


**Down Sampling**

In [33]:
# ============================================
# Downsampling (Undersampling) an Imbalanced Dataset
# ============================================

# Import the required libraries
import numpy as np
import pandas as pd

# Import the resample function from scikit-learn
# It is used to randomly sample data with or without replacement.
from sklearn.utils import resample

# --------------------------------------------
# Step 1: Set the random seed for reproducibility
# --------------------------------------------
# Ensures that the randomly generated data is the same
# every time the code is executed.
np.random.seed(123)

# --------------------------------------------
# Step 2: Create an imbalanced dataset
# --------------------------------------------

# Total number of samples
n_samples = 1000

# Proportion of samples belonging to Class 0 (majority class)
class_0_ratio = 0.9

# Calculate the number of samples in each class
n_class_0 = int(n_samples * class_0_ratio)   # 900 samples
n_class_1 = n_samples - n_class_0            # 100 samples

# Create the majority class (Class 0)
class_0 = pd.DataFrame({

    # Generate Feature 1 values from a normal distribution
    # with mean = 0 and standard deviation = 1.
    'feature_1': np.random.normal(loc=0, scale=1, size=n_class_0),

    # Generate Feature 2 values from the same distribution.
    'feature_2': np.random.normal(loc=0, scale=1, size=n_class_0),

    # Assign the target label 0 to all samples.
    'target': [0] * n_class_0
})

# Create the minority class (Class 1)
class_1 = pd.DataFrame({

    # Generate Feature 1 values from a normal distribution
    # with mean = 2 and standard deviation = 1.
    'feature_1': np.random.normal(loc=2, scale=1, size=n_class_1),

    # Generate Feature 2 values from the same distribution.
    'feature_2': np.random.normal(loc=2, scale=1, size=n_class_1),

    # Assign the target label 1 to all samples.
    'target': [1] * n_class_1
})

# --------------------------------------------
# Step 3: Combine both classes into one dataset
# --------------------------------------------

# Stack both DataFrames vertically and reset the index.
df = pd.concat([class_0, class_1]).reset_index(drop=True)

# Display the class distribution.
df['target'].value_counts()

# --------------------------------------------
# Step 4: Separate the majority and minority classes
# --------------------------------------------

# Select all majority class samples (target = 0).
df_majority = df[df['target'] == 0]

# Select all minority class samples (target = 1).
df_minority = df[df['target'] == 1]

# --------------------------------------------
# Step 5: Downsample the majority class
# --------------------------------------------

# Randomly select samples from the majority class
# without replacement to match the size of the minority class.
df_majority_downsampled = resample(
    df_majority,
    replace=False,                     # Do not duplicate samples
    n_samples=len(df_minority),        # Reduce majority class to 100 samples
    random_state=42                    # Ensure reproducibility
)

# --------------------------------------------
# Step 6: Combine the downsampled majority class
# with the original minority class
# --------------------------------------------

# Create a balanced dataset.
df_downsampled = pd.concat(
    [df_majority_downsampled, df_minority]
).reset_index(drop=True)

# --------------------------------------------
# Step 7: Verify the new class distribution
# --------------------------------------------

# Count the number of samples in each class.
print(df_downsampled['target'].value_counts())

target
0    100
1    100
Name: count, dtype: int64
